In [ ]:
import pandas as pd
import torch
from DownstreamModel.MLPHead import MLPHead
from transformers import AutoTokenizer, AutoModel , TrainingArguments, Trainer


In [2]:
from core.EssayDataset import EssayDataset
from core.BertBase import BertBasedNetwork

In [3]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
downstream = MLPHead(output_dim = 6)
model = BertBasedNetwork(downstream)


/Users/kimyingwong/anaconda3/envs/CSE780/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/Users/kimyingwong/anaconda3/envs/CSE780/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [4]:
essay = "Many people believe that cars bring convenience, but they also cause pollution and accidents."
enc = tokenizer(essay, return_tensors="pt", padding=True, truncation=True)

pred = model.single_batch_predict(enc)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [5]:
train_df = pd.read_csv("./learning-agency-lab-automated-essay-scoring-2/train.csv")

In [6]:
essay = train_df["full_text"][0]
enc = tokenizer(essay, return_tensors="pt", padding=True, truncation=True)

pred = model.single_batch_predict(enc)

In [7]:
# sampling for prediction 
# Include all rare scores
# Score 6: Keep all
rare_6 = train_df[train_df["score"] == 6].apply(
    lambda x: x.sample(frac=0.5, random_state=42)
)

# Score 1 & 5: Keep 50%
rare_1_5 = train_df[train_df["score"].isin([1, 5])].groupby("score", group_keys=False).apply(
    lambda x: x.sample(frac=0.05, random_state=42)
)

# Scores 2, 3, 4: Sample 15%
common = train_df[train_df["score"].isin([2, 3, 4])].groupby("score", group_keys=False).apply(
    lambda x: x.sample(frac=0.02, random_state=42)
)

# Combine all sampled subsets
selected_train_df = pd.concat([rare_6, rare_1_5, common]).reset_index(drop=True)




In [8]:
selected_train_df

,essay_id,full_text,score
0,a1cd4ce,Our country's history with cars date back a fe...,6
1,77cb526,"The ""Facial Action Coding System"" or ""FACS"" is...",6
2,86f807f,"In ""The Challeneges of Exploring Venus"", the a...",6
3,7fe2d12,"To the state and the state's country, the elec...",6
4,b8103da,"Dear Florida State Senator,\n\nEvery 4 years, ...",6
...,...,...,...
483,5df9875,Cars are a basic need for people today we use ...,4
484,4c28b83,"Every day people wake up, get ready, and go to...",4
485,09da56a,Over the years the amount of cars that have be...,4
486,95285d8,Their are people that think the fece on mars w...,4


In [10]:
from sklearn.model_selection import train_test_split



/Users/kimyingwong/anaconda3/envs/CSE780/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/kimyingwong/anaconda3/envs/CSE780/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):


In [11]:
selected_train_df["essay_id"] = pd.factorize(selected_train_df["essay_id"])[0] + 1
df_train, df_val = train_test_split(
    selected_train_df,
    test_size=0.2,         
    random_state=42,       
    stratify=selected_train_df["score"])

train_dataset = EssayDataset(df_train)
val_dataset = EssayDataset(df_val)

/Users/kimyingwong/anaconda3/envs/CSE780/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/kimyingwong/anaconda3/envs/CSE780/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
/Users/kimyingwong/anaconda3/envs/CSE780/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown to

In [11]:
train_dataset

In [ ]:
downstream = MLPHead(output_dim = 6)
model = BertBasedNetwork(downstream)



In [ ]:
training_args = TrainingArguments(
        output_dir="./essay_cls",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="qwk",
        greater_is_better=True,
        logging_dir="./logs",
        report_to="none",   # 避免用 wandb
    )

4.51.3


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=train_dataset.tokenizer,   
    compute_metrics=compute_metrics,
)
